In [1]:
import polars as pl
from math import sqrt, log
from collections import defaultdict
from heapq import heapify, heappush, heappop,heapreplace
from tqdm.notebook import tqdm

In [2]:
articles_path='../data/articles.parquet'
transaction_path='../data/transactions.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [4]:
articles.shape

(105542, 12)

In [5]:
articles.head()

article_id,product_code,product_type_no,graphical_appearance_no,colour_group_code,perceived_colour_value_id,perceived_colour_master_id,department_no,index_code,index_group_no,section_no,garment_group_no
i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64
108775015,108775,253,1010016,9,4,5,1676,"""A""",1,16,1002
108775044,108775,253,1010016,10,3,9,1676,"""A""",1,16,1002
108775051,108775,253,1010017,11,1,9,1676,"""A""",1,16,1002
110065001,110065,306,1010016,9,4,5,1339,"""B""",1,61,1017
110065002,110065,306,1010016,10,3,9,1339,"""B""",1,61,1017


In [6]:
item_users_set=defaultdict(set)

In [7]:
transactions.shape

(31788324, 5)

In [8]:
transactions.head()

customer_id,article_id,price,sales_channel_id,time
str,i64,f64,u8,f64
"""000058a12d5b43e67d225668fa1f8d…",663713001,0.050831,2,1.5374e9
"""000058a12d5b43e67d225668fa1f8d…",541518023,0.030492,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",505221004,0.015237,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",685687003,0.016932,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",685687004,0.016932,2,1.5374e9


In [9]:
group_len=transactions.group_by('customer_id').len()['len'].to_numpy()

In [10]:
max(group_len)

1895

In [11]:
group_len.shape[0]

1362281

In [12]:
sum(group_len==1)

131514

In [13]:
for i in range(5,100+1,10):
    print(f'users count times >= {i} :',sum(group_len>=i))

users count times >= 5 : 925558
users count times >= 15 : 534761
users count times >= 25 : 367845
users count times >= 35 : 267297
users count times >= 45 : 200804
users count times >= 55 : 154672
users count times >= 65 : 121143
users count times >= 75 : 96220
users count times >= 85 : 77342
users count times >= 95 : 63128


观察到有的用户交互次数太大，做相似度计算时过于耗费时间，因此选择截断，只保留最近的30个

In [6]:
def build_itemcf(transactions,topk=50):
    item_cnt = defaultdict(int)
    cooc = defaultdict(float)

    for _, g in tqdm(transactions.group_by("customer_id"),total=transactions['customer_id'].unique().shape[0]):
        g = g.sort("time", descending=True).head(25)
        items = g.select(["article_id", "time"]).unique().to_numpy()
        for i, ti in items:
            item_cnt[i] += 1
            for j, tj in items:
                if i == j:
                    continue
                dt = abs(ti - tj) # 购买两个物品的间隔时间
                time_w = 1 / (1 + dt / (7 * 86400)) # 约束，在相邻时间内购买的物品，应有较大的权重
                cooc[(i, j)] += time_w

    item_sim=defaultdict(list)
    for (i, j), cij in cooc.items():

        weight=cij / (
            sqrt(item_cnt[i] * item_cnt[j]) * log(item_cnt[i] + 10)
        )

        if len(item_sim[i]) < topk:
            heappush(item_sim[i], (weight, j))
        else:
            if weight> item_sim[i][0][0]:
                heapreplace(item_sim[i], (weight, j))

    return item_sim

In [ ]:
def recall_itemcf(transactions, item_sim, topk=50, max_hist=10):
    res = {}

    user_hist = (
        transactions
        .group_by("customer_id")
        .agg([
            pl.col("article_id"),
            pl.col("time")
        ])
    )

    for row in tqdm(user_hist.iter_rows(named=True),total=transactions['customer_id'].unique().shape[0]):
        cid = row["customer_id"]

        # 按时间从近到远排序，并只取最近 max_hist 条
        hist = sorted(
            zip(row["article_id"], row["time"]),
            key=lambda x: x[1],
            reverse=True
        )[:max_hist]

        max_time = hist[0][1]
        hist_items = {i for i, _ in hist}

        score = defaultdict(float)

        for i, ti in hist:
            w_ui = 1 / (1 + (max_time - ti) / (7 * 86400))

            for j, s in item_sim.get(i, [])[:5]:
                if j in hist_items:
                    continue

                score[j] += s * w_ui

                # 候选数量已够，直接退出
                if len(score) >= topk:
                    break
            if len(score) >= topk:
                break

        res[cid] = [
            k for k, _ in
            sorted(score.items(), key=lambda x: -x[1])[:topk]
        ]

    return res


In [ ]:
def get_validation_data(data: pl.DataFrame):
    DAY = 86400
    WEEK = 7 * DAY

    max_time = data.select(pl.col("time").max()).item()

    valid_start = max_time - 6 * DAY
    train_start = valid_start - 6 * WEEK

    train_df = data.filter(
        (pl.col("time") >= train_start) &
        (pl.col("time") <  valid_start)
    )

    valid_df = data.filter(
        pl.col("time") >= valid_start
    )

    return train_df, valid_df

In [ ]:
def metric_recall(data,topk=5):
    train_df,valid_df=get_validation_data(data)
    item_sim=build_itemcf(data)

    user_item = recall_itemcf(train_df, item_sim,topk * 10)

    for k in range(10,topk*10+1,10):
        total_recall = 0.0
        user_cnt = 0
        for cid,group in valid_df.group_by('customer_id'):
            group=set(group['article_id'])
            if cid not in user_item:
                continue
            pred_item=set(user_item[cid][:k])
            total_recall+=len(pred_item&group)/len(group)
            user_cnt+=1
        avg_recall=total_recall/user_cnt
        print(f"Recall@{k}: {avg_recall:.6f}")

In [ ]:
item_sim=build_itemcf(transactions)
user_item = recall_itemcf(transactions, item_sim,100)
import pickle

with open("../save/candidate/item_sim.pkl", "wb") as f:
    pickle.dump(item_sim, f, protocol=pickle.HIGHEST_PROTOCOL)

rows = []
for cid, items in user_item.items():
    for rank, aid in enumerate(items):
        rows.append((cid, aid, rank))

df_user_item = pl.DataFrame(
    rows,
    schema=["customer_id", "article_id", "rank"]
)
df_user_item.write_parquet("../save/candidate/recall_itemcf.parquet")

In [ ]:
metric_recall(transactions)